In [1]:
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from google.oauth2.service_account import Credentials

In [2]:
def connect_to_gsheet(creds_json,spreadsheet_name):
    scope = ["https://spreadsheets.google.com/feeds", 'https://www.googleapis.com/auth/spreadsheets',
             "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]
    
    credentials = ServiceAccountCredentials.from_json_keyfile_name(creds_json, scope)
    client = gspread.authorize(credentials)
    spreadsheet = client.open(spreadsheet_name)  # Access the first sheet
    return spreadsheet

In [3]:
SPREADSHEET_NAME = 'Offline Data'
# SPREADSHEET_NAME_2 = 'Tab 2 Data'
# SHEET_NAME = 'Sheet1'
CREDENTIALS_FILE = './private_key.json'
sheet_by_name = connect_to_gsheet(CREDENTIALS_FILE, SPREADSHEET_NAME)

In [4]:
ws = sheet_by_name.worksheet("interaction_data")
x=ws.get_all_records()
int_data=pd.DataFrame(x)
int_data

,user_id,item_id
0,33,32
1,27,8
2,4,46
3,33,8
4,18,43
...,...,...
995,41,44
996,29,44
997,27,24
998,14,38


In [6]:
ws = sheet_by_name.worksheet("user_data")
x=ws.get_all_records()
user_data=pd.DataFrame(x)

ws = sheet_by_name.worksheet("item_data")
x=ws.get_all_records()
item_data=pd.DataFrame(x)

In [5]:
int_data['label']=1
int_data

,user_id,item_id,label
0,33,32,1
1,27,8,1
2,4,46,1
3,33,8,1
4,18,43,1
...,...,...,...
995,41,44,1
996,29,44,1
997,27,24,1
998,14,38,1


In [25]:
all_items=set(item_data['item_id'].unique())
negative_samples = []
for user in int_data['user_id'].unique():
    interacted_items = set(int_data[int_data["user_id"] == user]["item_id"])
    non_interacted = list(all_items - interacted_items)

    sampled_items = np.random.choice(
        non_interacted,
        size=min(2, len(non_interacted)),
        replace=False
    )

    for item in sampled_items:
        negative_samples.append([user, item, 0])

neg_df = pd.DataFrame(negative_samples, columns=["user_id", "item_id", "label"])

df=pd.concat([int_data, neg_df], ignore_index=True)
df

,user_id,item_id,label
0,33,32,1
1,27,8,1
2,4,46,1
3,33,8,1
4,18,43,1
...,...,...,...
1095,21,13,0
1096,23,24,0
1097,23,29,0
1098,24,32,0


In [26]:
df=df.merge(user_data, on='user_id', how='left')
df=df.merge(item_data, on='item_id', how='left')
df

,user_id,item_id,label,user_total_interactions,user_unique_items,user_unique_categories,user_avg_price,user_avg_rating,user_price_std,user_rating_std,item_total_interactions,item_unique_users,item_avg_price,item_avg_rating,item_price_std,item_rating_std,item_category,item_price,item_rating
0,33,32,1,22,18,5,234.147727,3.045455,131.986192,1.120258,23,17,88.52,4.8,0,0,Books,88.52,4.8
1,27,8,1,20,16,5,227.624500,2.605000,166.429260,1.065475,23,19,350.83,3.7,0,0,Books,350.83,3.7
2,4,46,1,22,17,5,281.930000,2.554545,140.697747,1.213702,21,13,412.60,1.3,0,0,Clothing,412.60,1.3
3,33,8,1,22,18,5,234.147727,3.045455,131.986192,1.120258,23,19,350.83,3.7,0,0,Books,350.83,3.7
4,18,43,1,20,17,5,249.123000,3.040000,144.079853,1.210220,21,17,5.77,2.5,0,0,Electronics,5.77,2.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,21,13,0,20,17,5,202.276500,2.330000,128.561864,0.872444,18,15,248.48,2.7,0,0,Toys,248.48,2.7
1096,23,24,0,22,18,5,275.124546,3.045455,131.870916,1.102261,23,16,494.35,1.4,0,0,Home,494.35,1.4
1097,23,29,0,22,18,5,275.124546,3.045455,131.870916,1.102261,23,21,236.57,1.9,0,0,Toys,236.57,1.9
1098,24,32,0,19,15,5,219.323684,2.778947,94.194139,1.210867,23,17,88.52,4.8,0,0,Books,88.52,4.8


In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
# import mlflow
# import mlflow.sklearn
		
X = df[["user_id", "item_id", "user_total_interactions","user_unique_items","user_unique_categories","user_avg_price","user_avg_rating","user_price_std",
       "user_rating_std","item_total_interactions","item_unique_users","item_avg_price","item_avg_rating","item_price_std","item_rating_std",
       "item_price","item_rating"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

X_test_copy = X_test.copy()
X_test_copy["label"] = y_test
X_test_copy["score"] = model.predict_proba(X_test)[:, 1]

def precision_at_k(df, k=5):
    df = df.sort_values(["user_id", "score"], ascending=[True, False])

    precision_list = []

    for user, group in df.groupby("user_id"):
        top_k = group.head(k)
        precision = top_k["label"].sum() / k
        precision_list.append(precision)

    return np.mean(precision_list)

precision = precision_at_k(X_test_copy, k=5)
precision

# -------------------------
# 5. MLflow Tracking
# -------------------------
# mlflow.set_experiment("recommender_mvp")

# with mlflow.start_run():

#     # Model
#     model = LogisticRegression(max_iter=200)
#     model.fit(X_train, y_train)

#     # Predictions
#     X_test_copy = X_test.copy()
#     X_test_copy["label"] = y_test
#     X_test_copy["score"] = model.predict_proba(X_test)[:, 1]

#     # Evaluation
#     precision = precision_at_k(X_test_copy, k=5)

#     # Logging
#     mlflow.log_param("model", "logistic_regression")
#     mlflow.log_param("features", "user_id,item_id,item_popularity,user_activity")
#     mlflow.log_metric("precision_at_5", precision)

#     mlflow.sklearn.log_model(model, "model")

#     print(f"Precision@5: {precision:.4f}")

C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


np.float64(0.7306122448979593)